In [7]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt 

data = pd.read_csv('train.csv')

In [8]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T #transponerer tabellen, slik at labels ligger i rad, ikke kolonne
Y_dev = data_dev[0] #plukker ut hele rad 0, som er selve tallene som skal gjettes
X_dev = data_dev[1:n] #i resterende rader, fra rad 1 og oppover, likker pikslene til hver label
X_dev = X_dev / 255. #normaliserer pikslene slik at de ligger mellom 0 og 1, og ikke mellom 0 og 255.

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.
_,m_train = X_train.shape

In [9]:
Y_train

array([5, 8, 7, ..., 1, 0, 7], shape=(41000,))

In [ ]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5 #Vekter for første lag, 10 noder, 784 input features (28*28 piksler)
    b1 = np.random.rand(10, 1) - 0.5 #Bias for første lag, 10 noder
    W2 = np.random.rand(10, 10) - 0.5 #Vekter for andre lag, 10 noder, 10 output classes (digits 0-9)
    b2 = np.random.rand(10, 1) - 0.5 #Bias for andre lag, 10 noder
    return W1, b1, W2, b2

def ReLU(z):
    return np.maximum(z,0) #Det returnerer z hvis z er større enn 0, ellers returnerer det 0. Dette er ReLU-aktiveringsfunksjonen.

def softmax(z):
    A = np.exp(z) / sum(np.exp(z)) #Softmax funker slik at alle verdiene i A summerer til 1, og kan dermed tolkes som sannsynligheter.
    return A #Returnerer A, som er en vektor med sannsynligheter for hver klasse.

def forward_prop(W1, b1, W2, b2, X): #Forward propagation, som tar inn vekter, bias og input data X. Propagation betyr at vi sender input data gjennom nettverket for å få output. 
    Z1 = W1.dot(X) + b1 #Z1 er summen av vektene multiplisert med input data X,pluss bias b1. Dette er input til første lag. 
    A1 = ReLU(Z1) #A1 er output fra første lag, som er resultatet av å bruke ReLU-aktiveringsfunksjonen på Z1. Dette er input til andre lag. 
    Z2 = W2.dot(A1) + b2 #Z1 er summen av vektene i andre lag multiplisert med output fra første lag A1, pluss bias b2. Dette er input til andre lag.
    A2 = softmax(Z2) #A2 er output fra andre lag, som er resultatet av å bruke softmax-aktiveringsfunksjonen på Z2. Dette er output fra hele nettverket, som representerer sannsynligheten for hver klasse (0-9) gitt input data X. 
    return Z1, A1, Z2, A2

def ReLU_deriv(Z): #Derivert av ReLU-aktiveringsfunksjonen. Returnerer en matrise med samme dimensjoner som Z, der elementene er 1 hvis Z > 0, ellers 0. Dette brukes i backpropagation for å beregne gradientene. 
    return Z > 0

def one_hot(Y): #Konverterer labels Y til one-hot encoding. One-hot encoding er en måte å representere kategoriske variabler som binære vektorer. 
    one_hot_Y = np.zeros((Y.size, Y.max() + 1)) 
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y): #Backward propagation, som tar inn output fra forward propagation, vekter, input data X og labels Y. Dette brukes for å beregne gradientene for vektene og biasene i nettverket. 
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2




In [ ]:
def get_predictions(A2): #Henter de predikerte klassene basert på sannsynlighetene i A2. A2 er output fra softmax-funksjonen, som gir sannsynligheter for hver klasse. 
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y): #Henter nøyaktigheten til prediksjonene ved å sammenligne dem med de faktiske labels Y.
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations): #Gradient descent er en optimaliseringsalgoritme som brukes for å minimere en funksjon ved å iterativt bevege seg i retning av den bratteste nedstigningen som definert av den negative gradienten. I dette tilfellet brukes en gradient descent algoritme for å oppdatere vektene og biasene i et nevralt nettverk basert på gradientene beregnet fra backpropagation.
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 500) #This line runs the gradient descent on the training data for 500 iterations whith a learning rate of 0.1. It returns the final weights and biases for both layers of the neural network. 

In [ ]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

def test_prediction(index, W1, b1, W2, b2):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2)
    label = Y_train[index]
    print("Prediction: ", prediction)
    print("Label: ", label)

    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

In [ ]:
test_prediction(0, W1, b1, W2, b2)
test_prediction(1, W1, b1, W2, b2)
test_prediction(2, W1, b1, W2, b2)
test_prediction(3, W1, b1, W2, b2)

In [ ]:
dev_predictions = make_predictions(X_dev, W1, b1, W2, b2)
get_accuracy(dev_predictions, Y_dev)